# 02 · Feature engineering

**Objective:** Create a leakage-conscious, daily QQQ prediction dataset.

**Input:** `data/qqq_prices.csv` from notebook 01.

**Output:** `data/qqq_model_data.csv` with four features and the
**next** trading day's close-to-close return as the target.
All feature values for date t use information available by the close
of date t; they do not include tomorrow's price.


## 1. Load the fixed historical price snapshot

Using the saved source data avoids repeated downloads and keeps feature
generation consistent with the other notebooks.


In [1]:
from pathlib import Path
import sys

import pandas as pd

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from src.qqq_utils import FEATURE_COLUMNS, TARGET_COLUMN

price_path = project_dir / "data" / "qqq_prices.csv"
if not price_path.exists():
    raise FileNotFoundError("Run notebook 01 first to save data/qqq_prices.csv")
qqq = pd.read_csv(price_path, index_col="Date", parse_dates=["Date"])
assert qqq.index.is_monotonic_increasing and not qqq.index.has_duplicates
print("Prices loaded:", qqq.shape, "dates", qqq.index.min().date(),
      "to", qqq.index.max().date())


Prices loaded: (4191, 5) dates 2010-01-04 to 2026-09-01


## 2. Create the four predictors

All values are percentages (or percentage-point volatility):

| Feature | Meaning | Information available |
|---|---|---|
| `Daily_Return_Pct` | Today's close-to-close return | Close t and close t−1 |
| `Momentum_5d_Pct` | Five-session price return | Close t and close t−5 |
| `Volatility_20d_Pct` | Standard deviation of 20 daily returns | Returns through t |
| `MA_Distance_20d_Pct` | Percent distance above/below 20-day average | Prices through t |

**Today's daily return is an X feature.** It is distinct from the
**tomorrow** return we are predicting. Momentum and moving-average
features share price information; they are not independent variables.


In [2]:
qqq["Daily_Return_Pct"] = qqq["Close"].pct_change() * 100
qqq["Momentum_5d_Pct"] = qqq["Close"].pct_change(periods=5) * 100
qqq["Volatility_20d_Pct"] = qqq["Daily_Return_Pct"].rolling(20).std()
qqq["SMA_20d"] = qqq["Close"].rolling(20).mean()
qqq["MA_Distance_20d_Pct"] = (qqq["Close"] / qqq["SMA_20d"] - 1) * 100

qqq[["Close"] + FEATURE_COLUMNS].iloc[18:25]


,Close,Daily_Return_Pct,Momentum_5d_Pct,Volatility_20d_Pct,MA_Distance_20d_Pct
Date,,,,,
2010-01-29,37.060711,-1.745111,-3.102295,NaN,NaN
2010-02-01,37.467773,1.098367,-2.369660,NaN,-4.666398
2010-02-02,37.805565,0.901552,-1.578292,1.265404,-3.512421
2010-02-03,38.013435,0.549841,-1.812020,1.278162,-2.709847
2010-02-04,36.913464,-2.893639,-2.135491,1.405651,-5.155049
2010-02-05,37.225269,0.844694,0.444024,1.429443,-4.013200
2010-02-08,36.956776,-0.721267,-1.363833,1.404376,-4.290857


## 3. Define tomorrow's target

`shift(-1)` pairs the row-t predictors with the **return from close t
to close t+1**. It does not feed tomorrow's return into the predictors.
The final row has no observable next-day target and will be dropped.


In [3]:
qqq[TARGET_COLUMN] = qqq["Daily_Return_Pct"].shift(-1)

# Check that today's target equals the next row's daily return.
assert qqq[TARGET_COLUMN].iloc[:-1].equals(
    qqq["Daily_Return_Pct"].iloc[1:].set_axis(qqq.index[:-1])
)
qqq[["Daily_Return_Pct", TARGET_COLUMN]].head(6)


,Daily_Return_Pct,Target_Next_Day_Return_Pct
Date,,
2010-01-04,NaN,0.000000
2010-01-05,0.000000,-0.603184
2010-01-06,-0.603184,0.065026
2010-01-07,0.065026,0.822979
2010-01-08,0.822979,-0.408112
2010-01-11,-0.408112,-1.251089


## 4. Remove only rows that cannot be modeled

Trailing indicators need a warm-up period. We drop those early rows
and the last row without a future label *together*, so feature/target
dates remain aligned. The eventual train/validation/test splits are
chronological and are constructed in notebook 03.


In [4]:
model_data = qqq[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
print("Rows before removing missing values:", len(model_data))
print("Missing by column:")
print(model_data.isna().sum())

model_data = model_data.dropna().copy()
assert not model_data.isna().any().any()
assert model_data.index.is_monotonic_increasing
assert not model_data.index.has_duplicates

print("Model-ready rows:", len(model_data))
print("Dates:", model_data.index.min().date(), "to",
      model_data.index.max().date())
model_data.head()


Rows before removing missing values: 4191
Missing by column:
Daily_Return_Pct               1
Momentum_5d_Pct                5
Volatility_20d_Pct            20
MA_Distance_20d_Pct           19
Target_Next_Day_Return_Pct     1
dtype: int64
Model-ready rows: 4170
Dates: 2010-02-02 to 2026-08-31


,Daily_Return_Pct,Momentum_5d_Pct,Volatility_20d_Pct,MA_Distance_20d_Pct,Target_Next_Day_Return_Pct
Date,,,,,
2010-02-02,0.901552,-1.578292,1.265404,-3.512421,0.549841
2010-02-03,0.549841,-1.812020,1.278162,-2.709847,-2.893639
2010-02-04,-2.893639,-2.135491,1.405651,-5.155049,0.844694
2010-02-05,0.844694,0.444024,1.429443,-4.013200,-0.721267
2010-02-08,-0.721267,-1.363833,1.404376,-4.290857,1.031183


## 5. Save the reproducible modeling dataset

Every later notebook reads this exact CSV. We do not re-engineer
indicators in each new notebook, avoiding multiple definitions of the
same feature or target.


In [5]:
# Save the completed modeling dataset
model_path = project_dir / "data" / "qqq_model_data.csv"

model_data.to_csv(model_path, index_label="Date")

print("Dataset saved:", model_path)
print("Observations:", len(model_data))

Dataset saved: /Users/michaelsipahimalani/Projects/qqq-polished/data/qqq_model_data.csv
Observations: 4170


## Takeaway and limitation

Each row is one prediction made **after its day's close**. Because the
target is a next-day **close-to-close** return, the resulting model is
fundamentally a return-forecasting experiment. That target is not the
same as a next-day open-to-close trade return; notebook 05 analyzes
this execution mismatch explicitly.
